# Lab 8 · Notebook 2: Real, or chance? Tests with scipy

**Time:** about 40 minutes. **What you will do:** put a number on the differences you have been looking at. You will see what chance alone does, run a Mann–Whitney test, decide whether a result matters, write a `compare()` function and run it on many genes. At the end, a correlation and a regression line.

This is the toolkit your project needs: one test, one function, and the habit of reading the **difference first, then p**.

Every **Write it** cell has a self-check underneath. Keep the variable names the task gives you.

In [ ]:
# Run this first. It loads the DepMap table and makes skin and rest.
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import linregress, mannwhitneyu, pearsonr, spearmanr

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab08/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]
print("all:", len(df), "  skin:", len(skin), "  rest:", len(rest))

## 1. Chance makes differences too

Skin lines need SOX10 far more than the rest. But **any** two groups of cell lines have different medians. The question is whether the skin difference is bigger than a random split would give.

### Write it

**Your task:** store the median SOX10 effect of skin **minus** the median of the rest in `skin_diff`.

In [ ]:
# TODO - the real difference: skin median minus rest median
skin_diff = skin["SOX10"].median() - rest["SOX10"].median()
print(round(skin_diff, 2))

# --- self-check (do not edit) ---
assert abs(skin_diff - (skin["SOX10"].median() - rest["SOX10"].median())) < 1e-9, "Use .median() on skin[\"SOX10\"] and rest[\"SOX10\"], skin first"
print("Correct: skin_diff =", round(skin_diff, 2))

Now the same thing for groups made by chance. The demo below picks 73 cell lines **at random** from the whole table, 500 times, and stores each fake difference. The histogram shows what chance can do. The red line is the real skin difference.

In [ ]:
# DEMO - 500 random groups of 73 cell lines
fake_diffs = []
for i in range(500):
    fake = df.sample(73)                       # 73 rows chosen at random
    others = df.drop(fake.index)               # everyone else
    fake_diffs.append(fake["SOX10"].median() - others["SOX10"].median())

fig, ax = plt.subplots()
sns.histplot(x=fake_diffs, bins=30, ax=ax)
ax.axvline(skin["SOX10"].median() - rest["SOX10"].median(), color="red", label="skin")
ax.set_xlabel("median difference, random group minus the rest")
ax.legend()
plt.show()
print("largest chance difference:", round(min(fake_diffs), 2), "to", round(max(fake_diffs), 2))

Chance moves the median by a few hundredths. The skin difference is far outside anything chance produced. That is exactly the question a test answers, without you having to run 500 shuffles: **if there were no real difference, how often would chance give one this big?** That probability is the p-value.

## 2. The Mann–Whitney test

`scipy.stats` has the tests. We use **Mann–Whitney U** to compare two groups. It works on ranks, so it does not mind a long tail or a skewed shape, and it is the safe choice when you are unsure how your data are distributed.

```python
result = mannwhitneyu(a, b)
result.statistic    # the U statistic, rarely needed
result.pvalue       # the number you report
```

In [ ]:
# DEMO - skin vs the rest, SOX10
result = mannwhitneyu(skin["SOX10"], rest["SOX10"])
print("U =", result.statistic)
print("p =", result.pvalue)
print(f"p = {result.pvalue:.2g}")        # :.2g keeps 2 significant figures

`1.3e-27` means 1.3 × 10⁻²⁷: a 1 with 26 zeros in front of it after the point. Chance essentially never does that.

### Write it

**Your task:** test `BRAF`, skin vs the rest. Store the p-value (just the number) in `p_braf`.

In [ ]:
# TODO - Mann-Whitney for BRAF, skin vs rest
p_braf = mannwhitneyu(skin["BRAF"], rest["BRAF"]).pvalue
print(f"p = {p_braf:.2g}")

# --- self-check (do not edit) ---
assert isinstance(p_braf, float), f"p_braf is a {type(p_braf).__name__}. Store result.pvalue, not the whole result"
assert abs(p_braf - mannwhitneyu(skin["BRAF"], rest["BRAF"]).pvalue) < 1e-30, "That is not the BRAF p-value. Pass skin[\"BRAF\"] and rest[\"BRAF\"]"
print("Correct: BRAF is very significant too")

### Write it

**Predict first:** PAX8 is needed by kidney and ovarian cancers. Will skin differ from the rest?

**Your task:** test `PAX8` the same way and store the p-value in `p_pax8`. Then set `pax8_significant` to `True` or `False` by comparing `p_pax8` with 0.05.

In [ ]:
# TODO - Mann-Whitney for PAX8, and is it below 0.05?
p_pax8 = mannwhitneyu(skin["PAX8"], rest["PAX8"]).pvalue
pax8_significant = p_pax8 < 0.05
print(f"p = {p_pax8:.2g}", pax8_significant)

# --- self-check (do not edit) ---
assert abs(p_pax8 - mannwhitneyu(skin["PAX8"], rest["PAX8"]).pvalue) < 1e-12, "That is not the PAX8 p-value"
assert pax8_significant == (p_pax8 < 0.05), "pax8_significant should be the result of p_pax8 < 0.05"
print("Correct: p =", round(p_pax8, 2), "- no evidence that skin differs in PAX8")

A p-value above 0.05 does **not** prove the groups are the same. It says this data gives no evidence they differ.

### Broken

This cell should print `significant` when p is below 0.05. Run it, read the error, and fix it.

In [ ]:
# BROKEN - compare the p-value, not the whole result
result = mannwhitneyu(skin["MITF"], rest["MITF"])
if result.pvalue < 0.05:
    message = "significant"
else:
    message = "not significant"
print(message)


# --- self-check (do not edit) ---
assert message == "significant", "MITF is significant. Compare result.pvalue with 0.05"
print("Fixed: mannwhitneyu returns a result with two parts; .pvalue is the one to compare")

## 3. Significant is not the same as big

With 73 and 1,092 cell lines, a test can pick up a difference too small to matter. So always read two numbers: **the difference first** (is it big enough to care about?), **then p** (could it be chance?).

### Write it

**Your task:** for `NRAS`, store the median difference (skin minus rest) in `nras_diff` and the Mann–Whitney p-value in `nras_p`. Then set `nras_verdict` to one of these three strings:

- `"real and big"`
- `"real but small"`
- `"could be chance"`

For scale: SOX10 differs by about -1.2. A difference of about -1 separates "dies without it" from "fine without it".

In [ ]:
# TODO - NRAS: difference, p, and your verdict
nras_diff = skin["NRAS"].median() - rest["NRAS"].median()
nras_p = mannwhitneyu(skin["NRAS"], rest["NRAS"]).pvalue
print(round(nras_diff, 2), f"p = {nras_p:.2g}")
nras_verdict = "real but small"

# --- self-check (do not edit) ---
assert abs(nras_diff - (skin["NRAS"].median() - rest["NRAS"].median())) < 1e-9, "nras_diff should be the skin median minus the rest median"
assert abs(nras_p - mannwhitneyu(skin["NRAS"], rest["NRAS"]).pvalue) < 1e-12, "nras_p should be the Mann-Whitney p-value"
assert nras_verdict in ["real and big", "real but small", "could be chance"], "Set nras_verdict to one of the three strings"
assert nras_verdict == "real but small", f"Look again: p = {nras_p:.2g} is below 0.05, but the difference is {nras_diff:.2f}"
print("Correct: p < 0.05, but a difference of", round(nras_diff, 2), "is tiny next to SOX10's -1.2")

## 4. Write it once: `compare(a, b)`

You now do the same three steps for every gene: two medians, a difference, a test. Put them in a function that returns a **dictionary**, so each number has a name.

### Write it

**Your task:** write `compare(a, b)`. `a` and `b` are two Series of numbers. Return a dictionary with these keys:

| key | value |
|---|---|
| `"median_a"` | the median of `a` |
| `"median_b"` | the median of `b` |
| `"difference"` | median of `a` minus median of `b` |
| `"p"` | the Mann–Whitney p-value |
| `"significant"` | `True` if `p < 0.05` |

In [ ]:
# TODO - compare two groups, return a dictionary
def compare(a, b):
    p = mannwhitneyu(a, b).pvalue
    return {
        "median_a": a.median(),
        "median_b": b.median(),
        "difference": a.median() - b.median(),
        "p": p,
        "significant": p < 0.05,
    }

# --- self-check (do not edit) ---
r = compare(skin["SOX10"], rest["SOX10"])
assert isinstance(r, dict), f"compare returned {type(r).__name__}. Return a dictionary {{...}}"
missing = {"median_a", "median_b", "difference", "p", "significant"} - set(r)
assert not missing, f"The dictionary is missing {missing}"
assert abs(r["difference"] - (skin["SOX10"].median() - rest["SOX10"].median())) < 1e-9, "difference should be median of a minus median of b"
assert abs(r["p"] - mannwhitneyu(skin["SOX10"], rest["SOX10"]).pvalue) < 1e-30, "p should be mannwhitneyu(a, b).pvalue"
assert r["significant"] == True, "significant should be p < 0.05"
assert compare(skin["PAX8"], rest["PAX8"])["significant"] == False, "PAX8 should not be significant. Use p < 0.05"
print("Correct: SOX10 difference", round(r["difference"], 2), f"p = {r['p']:.2g}", "significant:", r["significant"])

## 5. Many genes, one table

A function pays off in a loop. Collect one dictionary per gene in a list, then turn the list into a DataFrame: each dictionary becomes a row.

### Write it

**Your task:** run `compare(skin[gene], rest[gene])` for every gene in `genes`. Add the gene name to each dictionary (`r["gene"] = gene`), collect them in a list, and make a DataFrame `results` from it, **sorted by `"difference"`**, smallest first.

In [ ]:
# TODO - one row per gene, sorted by difference
genes = ["SOX10", "BRAF", "MITF", "MDM2", "TP53", "KRAS", "NRAS", "PAX8", "RAN"]
rows = []
for gene in genes:
    r = compare(skin[gene], rest[gene])
    r["gene"] = gene
    rows.append(r)
results = pd.DataFrame(rows).sort_values("difference")
print(results[["gene", "difference", "p", "significant"]].round(3))

# --- self-check (do not edit) ---
assert isinstance(results, pd.DataFrame), "results should be a DataFrame: pd.DataFrame(rows)"
assert len(results) == len(genes), f"results has {len(results)} rows. Add one row per gene"
assert "gene" in results.columns, "Add the gene name to each dictionary: r[\"gene\"] = gene"
assert list(results["gene"])[:2] == ["SOX10", "BRAF"], "Sort with .sort_values(\"difference\"), smallest first"
assert list(results["gene"])[-1] == "TP53", "Sort with .sort_values(\"difference\"), smallest first"
print("Correct: nine genes, one table, read top to bottom")

Read the table top to bottom, difference first, then p:

- **SOX10, BRAF, MDM2:** big and significant. Skin needs them.
- **TP53:** big, significant, and **positive**. Skin lines grow better without p53, which means p53 is still active in most of them. Your project looks at p53 in patients.
- **KRAS, NRAS:** significant but small.
- **PAX8, RAN:** no evidence of a difference. RAN is essential everywhere, so there is nothing to compare.

## 6. Two columns: do they move together?

The test above compares **two groups** on one gene. A correlation asks something else: across the same cell lines, do **two columns** rise and fall together?

- `pearsonr(x, y)` measures a straight-line trend.
- `spearmanr(x, y)` works on ranks, so it does not mind a long tail. Unsure? Use Spearman.

Both return `.statistic` (r, from -1 to +1) and `.pvalue`. `sns.regplot` draws the dots plus the best straight line.

In [ ]:
# DEMO - BRAF and SOX10 in the 73 skin lines
print("Pearson r: ", round(pearsonr(skin["BRAF"], skin["SOX10"]).statistic, 2))
print("Spearman r:", round(spearmanr(skin["BRAF"], skin["SOX10"]).statistic, 2))

fig, ax = plt.subplots()
sns.regplot(data=skin, x="BRAF", y="SOX10", ax=ax)
ax.set_title("73 skin cancer lines")
plt.show()

Skin lines that need BRAF tend to need SOX10 too: they are the melanomas. A correlation says the two move **together**, not that one causes the other.

### Write it

**Your task:** MITF is the master switch of pigment cells, and SOX10 turns it on. In the skin lines, store the Spearman r of `MITF` vs `SOX10` in `r_mitf` (just the number), and draw a `sns.regplot` with MITF on x and SOX10 on y.

In [ ]:
# TODO - Spearman r and a regplot: MITF vs SOX10, skin only
r_mitf = spearmanr(skin["MITF"], skin["SOX10"]).statistic
print("Spearman r:", round(r_mitf, 2))

fig, ax = plt.subplots()
sns.regplot(data=skin, x="MITF", y="SOX10", ax=ax)
plt.show()

# --- self-check (do not edit) ---
assert abs(r_mitf - spearmanr(skin["MITF"], skin["SOX10"]).statistic) < 1e-9, "r_mitf should be spearmanr(skin[\"MITF\"], skin[\"SOX10\"]).statistic"
points = ax.collections[0].get_offsets()
assert len(points) == 73, f"Your plot has {len(points)} dots. Use data=skin"
assert ax.get_xlabel() == "MITF" and ax.get_ylabel() == "SOX10", "Put MITF on x and SOX10 on y"
print("Correct: r =", round(r_mitf, 2), "- a weaker trend than BRAF, but the same direction")

## 7. A line through the dots: linear regression

`sns.regplot` drew a straight line through the dots. `linregress` gives you the numbers behind it:

```python
fit = linregress(x, y)
fit.slope        # how much y changes when x goes up by 1
fit.intercept    # y where x is 0
fit.rvalue       # Pearson r; square it for r squared
fit.pvalue       # could the slope be 0, by chance?
```

The line is `y = intercept + slope * x`. **r squared** is the share of the spread in y that the line explains, from 0 (none) to 1 (all of it).

In [ ]:
# DEMO - the line behind the regplot: BRAF predicts SOX10 in skin
fit = linregress(skin["BRAF"], skin["SOX10"])
print("slope:    ", round(fit.slope, 2))
print("intercept:", round(fit.intercept, 2))
print("r squared:", round(fit.rvalue ** 2, 2))
print(f"p = {fit.pvalue:.2g}")

A slope of 0.7: for every 1 unit a skin line depends more on BRAF, it depends about 0.7 more on SOX10. An r squared of 0.42 means the line explains about 40 % of the spread in SOX10. The rest is scatter around the line.

### Write it

**Your task:** use `fit` from the demo.

1. Store r squared in `r_squared`.
2. A new skin line has a BRAF effect of **-1.0**. Use the line to predict its SOX10 effect and store it in `predicted`.

In [ ]:
# TODO - r squared, and a prediction from the line
r_squared = fit.rvalue ** 2
predicted = fit.intercept + fit.slope * -1.0
print("r squared:", round(r_squared, 2), "  predicted SOX10:", round(predicted, 2))

# --- self-check (do not edit) ---
assert abs(r_squared - fit.rvalue ** 2) < 1e-9, "r squared is fit.rvalue ** 2"
assert abs(predicted - (fit.intercept + fit.slope * -1.0)) < 1e-9, "Use the line: fit.intercept + fit.slope * x, with x = -1.0"
print("Correct: a skin line with BRAF -1.0 should have SOX10 near", round(predicted, 2))

### Write it

**Your task:** draw the line yourself. Make a scatter plot of the skin lines (BRAF on x, SOX10 on y) with `ax.scatter`, then draw the fitted line on the same `ax` with `ax.plot(xs, ys, color="red")`, where `xs` is two x values (the smallest and largest BRAF) and `ys` the line's y values at those points. Compare it with the regplot in section 6: it should be the same line.

In [ ]:
# TODO - scatter plus your own regression line
xs = [skin["BRAF"].min(), skin["BRAF"].max()]
ys = [fit.intercept + fit.slope * x for x in xs]

fig, ax = plt.subplots()
ax.scatter(skin["BRAF"], skin["SOX10"], s=12)
ax.plot(xs, ys, color="red")
ax.set_xlabel("BRAF gene effect")
ax.set_ylabel("SOX10 gene effect")
plt.show()

# --- self-check (do not edit) ---
import numpy as np
assert len(ax.collections) == 1 and len(ax.collections[0].get_offsets()) == 73, "Draw the 73 skin lines with ax.scatter"
assert len(ax.lines) == 1, f"The panel has {len(ax.lines)} lines. Draw one with ax.plot(xs, ys)"
lx, ly = ax.lines[0].get_data()
lx, ly = np.asarray(lx, dtype=float), np.asarray(ly, dtype=float)
assert np.allclose(ly, fit.intercept + fit.slope * lx), "The red line is not the fitted line. ys should be fit.intercept + fit.slope * x for each x in xs"
assert lx.min() <= skin["BRAF"].min() + 0.01 and lx.max() >= skin["BRAF"].max() - 0.01, "Let the line run from the smallest to the largest BRAF value"
print("Correct: your line is the regplot line")

Correlation and regression use the same line but ask different things. Spearman or Pearson r: **do** the two columns move together? The regression: **by how much**, so you can predict one from the other. Neither says that one causes the other.

## If you finish early

1. Run `compare` on **every** gene: `for gene in df.columns[4:]:`. Sort by difference and look at the 10 genes skin needs most. How many of the 257 come out significant?
2. Every test at 0.05 has a 1 in 20 chance of a false alarm, so 257 tests should give about 13 on pure chance. Try it: compare a random group of 73 lines (`df.sample(73)`) with the rest for every gene, and count the "significant" ones. If you want to know how scientists deal with this, look up *multiple testing correction* in the DataCamp statistics courses.
3. Use `group_figure` from notebook 1 to draw the three genes at the top of your table.
4. Fit `linregress` for MITF vs SOX10 in skin. Is r squared bigger or smaller than for BRAF? Does that match the Spearman r you found?
5. Want to compute p-values without any formula? Try the optional notebook 3: permutation tests and the bootstrap.